# WideResNet teacher → SE-ResNet student distillation
Train a WideResNet teacher from scratch, then distill it into an SE-ResNet student under 5M parameters. Only the student is submitted. The teacher runs for 150 epochs and the student for 200 epochs. Both use the same fixed stratified 80/20 split, seed, ID handling, and normalization as `cifar10_resnet_5m.ipynb`.

Run top to bottom on Kaggle with the competition data attached and a GPU enabled. Each phase saves a resumable checkpoint after every completed epoch, including its best weights. Re-running skips completed phases and resumes interrupted phases. A partial epoch repeats. Student predictions use ten-view test-time augmentation and produce `/kaggle/working/submission.csv` in sample order.

No fixed runtime is promised. Epoch logs include measured seconds and an approximate remaining time for the current phase, based on completed epochs in this session. Teacher and student speeds differ; their estimates are separate.

Enable Kaggle file persistence where available and preserve notebook outputs. For an attached checkpoint, set `TEACHER_RESUME_CHECKPOINT` or `STUDENT_RESUME_CHECKPOINT` to its path under `/kaggle/input`. A resumed student also requires the same trained teacher: preserve the teacher checkpoint too. `AUTO_RESUME=False` intentionally starts new runs and replaces outputs. Save & Run All launches a fresh background run; it resumes only if checkpoint files are available to it. Kaggle runtime and quota limits still apply.

Phase 1 first looks for `wrn_teacher.pt` in `/kaggle/working`, then attached inputs under `/kaggle/input`. If found, it loads and freezes the teacher without training. Otherwise it trains/resumes the teacher as configured. Attach the saved notebook outputs to make a previous commit available. An invalid or incompatible saved teacher raises an error rather than silently retraining. Legacy exports without split metadata require verifying that they used the same 80/20 split.


In [1]:
from pathlib import Path
import csv
import time
import hashlib
import torch.nn.functional as F
import os
import random
import numpy as np
from PIL import Image
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

SEED = 42
BATCH_SIZE = 128
LEARNING_RATE = 0.1
NUM_WORKERS = 2
AUTO_RESUME = True
# Optional: checkpoint attached as a Kaggle notebook output/dataset.
# Example: Path('/kaggle/input/my-saved-run/last_checkpoint.pt')
TEACHER_RESUME_CHECKPOINT = None
STUDENT_RESUME_CHECKPOINT = None
TEACH = dict(depth=28, widen=8, epochs=150)
STU = dict(blocks=[4, 4, 3], channels=[64, 128, 256], epochs=200)
T_KD = 4.0
ALPHA = 0.7
WARM = 5
CLIP = 5.0
INPUT_ROOT = Path('/kaggle/input/competitions/cifar-10-res-net-architecture-challenge-dl')
WORK_DIR = Path('/kaggle/working')
WORK_DIR.mkdir(parents=True, exist_ok=True)
LABELS_CSV = INPUT_ROOT / 'train_labels.csv'
TRAIN_IMAGE_DIR = INPUT_ROOT / 'train_images'

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

seed_everything(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

DEVICE = str(device)
assert device.type == 'cuda', 'Enable a GPU in Kaggle before training.'


Device: cuda


In [2]:
assert LABELS_CSV.is_file(), f'Missing labels CSV: {LABELS_CSV}'
assert TRAIN_IMAGE_DIR.is_dir(), f'Missing training image directory: {TRAIN_IMAGE_DIR}'
LABELS_CSV = Path(LABELS_CSV)
with LABELS_CSV.open(newline='') as f:
    reader = csv.DictReader(f)
    assert {'id', 'label'} <= set(reader.fieldnames or []), 'Expected id,label columns'
    rows = sorted(list(reader), key=lambda row: row['id'])
assert rows and len({r['id'] for r in rows}) == len(rows), 'IDs must be unique'
classes = sorted({r['label'] for r in rows})
assert len(classes) == 10, classes
class_to_idx = {name: i for i, name in enumerate(classes)}
labels = np.array([class_to_idx[r['label']] for r in rows], dtype=np.int64)
TRAIN_IMAGE_DIR = Path(TRAIN_IMAGE_DIR)
def training_image_path(image_id):
    # Accept numeric IDs, train_00001 IDs, or train_00001.png filenames.
    stem = Path(str(image_id).strip()).stem
    if stem.startswith('train_'):
        stem = stem[len('train_'):]
    if not stem.isdigit():
        raise ValueError(f'Unexpected training image ID: {image_id!r}')
    return TRAIN_IMAGE_DIR / f'train_{int(stem):05d}.png'

image_paths = [training_image_path(r['id']) for r in rows]
missing = [str(p) for p in image_paths if not p.is_file()]
assert not missing, f'Missing images, first examples: {missing[:5]}'

rng = np.random.default_rng(SEED)
train_indices, val_indices = [], []
for cls in range(10):
    indices = rng.permutation(np.flatnonzero(labels == cls))
    assert len(indices) % 5 == 0, 'Exact stratified 80/20 requires class sizes divisible by 5'
    n_val = len(indices) // 5
    val_indices.extend(indices[:n_val].tolist())
    train_indices.extend(indices[n_val:].tolist())
train_indices = np.array(sorted(train_indices))
val_indices = np.array(sorted(val_indices))
assert len(train_indices) * 5 == len(rows) * 4
assert len(val_indices) * 5 == len(rows)
assert not set(train_indices) & set(val_indices)
assert set(train_indices) | set(val_indices) == set(range(len(rows)))
with (WORK_DIR / 'split.csv').open('w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['id', 'label', 'split'])
    val_set = set(val_indices)
    for i, row in enumerate(rows):
        writer.writerow([row['id'], row['label'], 'validation' if i in val_set else 'training'])
print('Class mapping:', class_to_idx)
print('Training / validation:', len(train_indices), len(val_indices))
for cls, name in enumerate(classes):
    print(name, int((labels[train_indices] == cls).sum()), int((labels[val_indices] == cls).sum()))
# Fixed CIFAR normalization constants; no statistics are fitted on validation data.
MEAN = (0.4914, 0.4822, 0.4465)
STD = (0.2470, 0.2435, 0.2616)
train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.AutoAugment(transforms.AutoAugmentPolicy.CIFAR10),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
    transforms.RandomErasing(p=0.25),
])
val_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

class LabeledImages(Dataset):
    def __init__(self, indices, transform):
        self.indices = np.asarray(indices)
        self.transform = transform
    def __len__(self):
        return len(self.indices)
    def __getitem__(self, index):
        i = int(self.indices[index])
        with Image.open(image_paths[i]) as image:
            image = image.convert('RGB')
            assert image.size == (32, 32), f'Unexpected image size: {image.size}'
            x = self.transform(image)
        return x, int(labels[i])

train_dataset = LabeledImages(train_indices, train_transform)
val_dataset = LabeledImages(val_indices, val_transform)
assert torch.equal(val_dataset[0][0], val_dataset[0][0]), 'Validation must be deterministic'

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)

# A dedicated generator fixes training shuffle order across fresh runs.
train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, worker_init_fn=seed_worker,
    generator=torch.Generator().manual_seed(SEED),
    pin_memory=device.type == 'cuda',
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=device.type == 'cuda',
)


Class mapping: {'0': 0, '1': 1, '2': 2, '3': 3, '4': 4, '5': 5, '6': 6, '7': 7, '8': 8, '9': 9}
Training / validation: 40000 10000
0 4000 1000
1 4000 1000
2 4000 1000
3 4000 1000
4 4000 1000
5 4000 1000
6 4000 1000
7 4000 1000
8 4000 1000
9 4000 1000


In [3]:
# ---- Teacher: WideResNet (pre-activation) ----
class WRNUnit(nn.Module):
    def __init__(s,ic,oc,stride,drop=0.0):
        super().__init__(); s.bn1=nn.BatchNorm2d(ic); s.c1=nn.Conv2d(ic,oc,3,stride,1,bias=False)
        s.bn2=nn.BatchNorm2d(oc); s.c2=nn.Conv2d(oc,oc,3,1,1,bias=False); s.drop=drop
        s.equal=(ic==oc and stride==1); s.sc=None if s.equal else nn.Conv2d(ic,oc,1,stride,0,bias=False)
    def forward(s,x):
        if not s.equal: x=F.relu(s.bn1(x)); o=x
        else: o=F.relu(s.bn1(x))
        o=s.c1(o); o=F.relu(s.bn2(o))
        if s.drop>0: o=F.dropout(o,s.drop,training=s.training)
        o=s.c2(o); return o+(s.sc(x) if not s.equal else x)
class WideResNet(nn.Module):
    def __init__(s,depth=28,widen=8,nc=10,drop=0.0):
        super().__init__(); n=(depth-4)//6; w=[16,16*widen,32*widen,64*widen]
        s.conv1=nn.Conv2d(3,w[0],3,1,1,bias=False)
        def grp(ic,oc,stride):
            return nn.Sequential(*[WRNUnit(ic if i==0 else oc,oc,stride if i==0 else 1,drop) for i in range(n)])
        s.g1=grp(w[0],w[1],1); s.g2=grp(w[1],w[2],2); s.g3=grp(w[2],w[3],2)
        s.bn=nn.BatchNorm2d(w[3]); s.fc=nn.Linear(w[3],nc)
    def forward(s,x):
        o=s.conv1(x); o=s.g1(o); o=s.g2(o); o=s.g3(o)
        o=F.relu(s.bn(o)); return s.fc(F.adaptive_avg_pool2d(o,1).flatten(1))

# ---- Student: SE-ResNet [4,4,3] (<5M, submitted) ----
class SE(nn.Module):
    def __init__(s,ch,r=16):
        super().__init__(); d=max(ch//r,4); s.f1=nn.Linear(ch,d); s.f2=nn.Linear(d,ch)
    def forward(s,x): w=x.mean((2,3)); w=F.relu(s.f1(w)); w=torch.sigmoid(s.f2(w)); return x*w[:,:,None,None]
class SEBlock(nn.Module):
    def __init__(s,ic,oc,k=3,st=1):
        super().__init__(); p=k//2
        s.c1=nn.Conv2d(ic,oc,k,st,p,bias=False); s.b1=nn.BatchNorm2d(oc)
        s.c2=nn.Conv2d(oc,oc,k,1,p,bias=False); s.b2=nn.BatchNorm2d(oc); s.se=SE(oc)
        s.sc=nn.Sequential() if (st==1 and ic==oc) else nn.Sequential(nn.Conv2d(ic,oc,1,st,bias=False),nn.BatchNorm2d(oc))
    def forward(s,x): o=F.relu(s.b1(s.c1(x))); o=s.se(s.b2(s.c2(o))); return F.relu(o+s.sc(x))
class SENet(nn.Module):
    def __init__(s,bl,ch,stem=64,k=3,nc=10):
        super().__init__(); s.stem=nn.Sequential(nn.Conv2d(3,stem,3,1,1,bias=False),nn.BatchNorm2d(stem),nn.ReLU(True))
        L,ic=[],stem
        for i,(nb,oc) in enumerate(zip(bl,ch)):
            st=1 if i==0 else 2
            for b in range(nb): L.append(SEBlock(ic,oc,k,st if b==0 else 1)); ic=oc
        s.layers=nn.Sequential(*L); s.head=nn.Linear(ic,nc)
    def forward(s,x): x=s.stem(x); x=s.layers(x); return s.head(F.adaptive_avg_pool2d(x,1).flatten(1))
cp=lambda m: sum(p.numel() for p in m.parameters() if p.requires_grad)

In [4]:
def rbb(W, H, lam):
    r = np.sqrt(1 - lam)
    cw, ch = int(W * r), int(H * r)
    cx, cy = np.random.randint(W), np.random.randint(H)
    return (np.clip(cx-cw//2, 0, W), np.clip(cy-ch//2, 0, H),
            np.clip(cx+cw//2, 0, W), np.clip(cy+ch//2, 0, H))


def atomic_torch_save(payload, destination):
    destination = Path(destination)
    temporary = destination.with_name(destination.name + '.tmp')
    torch.save(payload, temporary)
    os.replace(temporary, destination)


def cpu_state(model):
    return {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}


def teacher_signature(model):
    # Bind a resumed student to the exact teacher weights, not just its architecture.
    digest = hashlib.sha256()
    for name, value in sorted(model.state_dict().items()):
        digest.update(name.encode())
        digest.update(value.detach().cpu().contiguous().numpy().tobytes())
    return digest.hexdigest()


@torch.inference_mode()
def vacc(model):
    model.eval()
    correct = count = 0
    for images, targets in val_loader:
        images, targets = images.to(device), targets.to(device)
        with torch.amp.autocast('cuda'):
            logits = model(images)
        correct += (logits.argmax(1) == targets).sum().item()
        count += targets.size(0)
    return correct / count


def train(model, epochs, model_config, teacher=None, mixup=False, ls=0.1,
          tag='teacher', resume_checkpoint=None):
    # Independent phase seeds make a fresh student reproducible even when a
    # completed teacher phase is loaded rather than trained in this session.
    seed_everything(SEED)
    train_loader.generator.manual_seed(SEED)
    ce = nn.CrossEntropyLoss(label_smoothing=ls)
    optimizer = torch.optim.SGD(model.parameters(), lr=LEARNING_RATE, momentum=0.9,
                                weight_decay=5e-4, nesterov=True)
    assert epochs > WARM
    scheduler = torch.optim.lr_scheduler.SequentialLR(optimizer, [
        torch.optim.lr_scheduler.LinearLR(optimizer, 0.1, 1.0, WARM),
        torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, epochs-WARM),
    ], [WARM])
    scaler = torch.amp.GradScaler('cuda')
    last_path = WORK_DIR / f'wrn_{tag}_last_checkpoint.pt'
    best_path = WORK_DIR / f'wrn_{tag}_best.pt'
    history_path = WORK_DIR / f'wrn_{tag}_history.csv'
    run_config = dict(
        tag=tag, epochs=epochs, model_config=model_config, seed=SEED,
        batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, learning_rate=LEARNING_RATE,
        classes=classes, ids=[row['id'] for row in rows],
        train_indices=train_indices.tolist(), val_indices=val_indices.tolist(),
        normalization=dict(mean=MEAN, std=STD), mixup=mixup, label_smoothing=ls,
        temperature=T_KD, alpha=ALPHA, warmup=WARM, gradient_clip=CLIP,
        teacher_signature=teacher_signature(teacher) if teacher is not None else None,
    )
    best_accuracy, best_epoch, start_epoch = -1.0, 0, 1
    best_checkpoint, history = None, []
    resume_path = Path(resume_checkpoint) if resume_checkpoint is not None else last_path
    if AUTO_RESUME and resume_path.is_file():
        saved = torch.load(resume_path, map_location='cpu', weights_only=True)
        assert saved.get('format_version') == 1, 'Expected a full WRN phase checkpoint'
        assert saved['run_config'] == run_config, (
            'Resume configuration, split, or teacher differs; restore original settings '
            'or set AUTO_RESUME=False for a fresh run.'
        )
        model.load_state_dict(saved['model_state'])
        optimizer.load_state_dict(saved['optimizer_state'])
        scheduler.load_state_dict(saved['scheduler_state'])
        scaler.load_state_dict(saved['scaler_state'])
        start_epoch = saved['epoch'] + 1
        best_accuracy, best_epoch = saved['best_accuracy'], saved['best_epoch']
        best_checkpoint, history = saved['best_checkpoint'], saved['history']
        atomic_torch_save(best_checkpoint, best_path)
        random.setstate(saved['python_rng_state'])
        state = saved['numpy_rng_state']
        np.random.set_state((state[0], np.asarray(state[1], dtype=np.uint32),
                             state[2], state[3], state[4]))
        torch.set_rng_state(saved['torch_rng_state'])
        train_loader.generator.set_state(saved['loader_rng_state'])
        if len(saved['cuda_rng_states']) == torch.cuda.device_count():
            torch.cuda.set_rng_state_all(saved['cuda_rng_states'])
        else:
            print('GPU count changed; exact CUDA random state cannot be restored.')
        print(f'[{tag}] Resuming after epoch {saved["epoch"]}; target {epochs}.', flush=True)
    else:
        if AUTO_RESUME and resume_checkpoint is not None:
            raise FileNotFoundError(resume_path)
        print(f'[{tag}] Starting a fresh {epochs}-epoch run.', flush=True)

    session_seconds = 0.0
    session_epochs = 0
    for epoch in range(start_epoch, epochs + 1):
        started = time.perf_counter()
        model.train()
        loss_sum, count = 0.0, 0
        learning_rate = optimizer.param_groups[0]['lr']
        for images, targets in train_loader:
            images = images.to(device, non_blocking=True)
            targets = targets.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            ya = yb = targets
            lam = 1.0
            if mixup:
                lam = np.random.beta(1.0, 1.0)
                idx = torch.randperm(images.size(0), device=device)
                ya, yb = targets, targets[idx]
                if random.random() < 0.5:
                    images = images.clone()
                    a, b, c, d = rbb(32, 32, lam)
                    images[:, :, b:d, a:c] = images[idx, :, b:d, a:c]
                    lam = 1 - ((c-a) * (d-b) / 1024)
                else:
                    images = lam * images + (1-lam) * images[idx]
            with torch.amp.autocast('cuda'):
                logits = model(images)
                if teacher is not None:
                    with torch.no_grad():
                        teacher_logits = teacher(images)
            hard_loss = lam * ce(logits.float(), ya) + (1-lam) * ce(logits.float(), yb)
            if teacher is not None:
                kd_loss = F.kl_div(
                    F.log_softmax(logits.float()/T_KD, dim=1),
                    F.softmax(teacher_logits.float()/T_KD, dim=1),
                    reduction='batchmean',
                ) * T_KD**2
                loss = ALPHA * kd_loss + (1-ALPHA) * hard_loss
            else:
                loss = hard_loss
            assert torch.isfinite(loss), 'Non-finite loss'
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), CLIP)
            scaler.step(optimizer)
            scaler.update()
            loss_sum += loss.detach().item() * targets.size(0)
            count += targets.size(0)
        scheduler.step()
        val_accuracy = vacc(model)
        if val_accuracy > best_accuracy:
            best_accuracy, best_epoch = val_accuracy, epoch
            best_checkpoint = dict(
                model_state=cpu_state(model), epoch=epoch, cfg=model_config,
                val_accuracy=val_accuracy, parameter_count=cp(model),
                classes=classes, seed=SEED,
                train_indices=train_indices.tolist(), val_indices=val_indices.tolist(),
                normalization=dict(mean=MEAN, std=STD),
            )
            atomic_torch_save(best_checkpoint, best_path)
        epoch_seconds = time.perf_counter() - started
        metrics = dict(epoch=epoch, learning_rate=learning_rate,
                       train_loss=loss_sum/count, val_accuracy=val_accuracy,
                       epoch_seconds=epoch_seconds)
        history.append(metrics)
        state = np.random.get_state()
        atomic_torch_save(dict(
            format_version=1, epoch=epoch, run_config=run_config,
            model_state=cpu_state(model), optimizer_state=optimizer.state_dict(),
            scheduler_state=scheduler.state_dict(), scaler_state=scaler.state_dict(),
            best_accuracy=best_accuracy, best_epoch=best_epoch,
            best_checkpoint=best_checkpoint, history=history,
            python_rng_state=random.getstate(),
            numpy_rng_state=(state[0], state[1].tolist(), int(state[2]),
                             int(state[3]), float(state[4])),
            torch_rng_state=torch.get_rng_state(),
            cuda_rng_states=torch.cuda.get_rng_state_all(),
            loader_rng_state=train_loader.generator.get_state(),
        ), last_path)
        # ETA includes checkpoint writing, not just GPU computation.
        elapsed = time.perf_counter() - started
        session_seconds += elapsed
        session_epochs += 1
        eta_minutes = (epochs-epoch) * session_seconds/session_epochs/60
        with history_path.open('w', newline='') as f:
            writer = csv.DictWriter(f, fieldnames=history[0].keys())
            writer.writeheader()
            writer.writerows(history)
        print(f'[{tag}] Epoch {epoch:03d}/{epochs}: lr={learning_rate:.6f}, '
              f'train loss={loss_sum/count:.4f}, val acc={val_accuracy:.2%}, '
              f'best={best_accuracy:.2%}, seconds={elapsed:.1f}, '
              f'phase ETA={eta_minutes:.1f} min', flush=True)
    assert best_checkpoint is not None, 'No best checkpoint available'
    model.load_state_dict(best_checkpoint['model_state'])
    model.eval()
    if history:
        with history_path.open('w', newline='') as f:
            writer = csv.DictWriter(f, fieldnames=history[0].keys())
            writer.writeheader()
            writer.writerows(history)
    print(f'[{tag}] Best validation accuracy: {best_accuracy:.2%} (epoch {best_epoch})')
    return best_accuracy


In [5]:
# Phase 1: reuse an exported teacher; otherwise train/resume it.
def find_saved_teacher(work_dir, input_root):
    local_path = Path(work_dir) / 'wrn_teacher.pt'
    if local_path.is_file():
        return local_path
    candidates = sorted(Path(input_root).rglob('wrn_teacher.pt'))
    if len(candidates) > 1:
        raise ValueError(
            f'Multiple saved teachers found: {candidates}. '
            'Attach only the intended teacher output, or copy it to '
            '/kaggle/working/wrn_teacher.pt.'
        )
    return candidates[0] if candidates else None


teacher_path = find_saved_teacher(WORK_DIR, Path('/kaggle/input'))
seed_everything(SEED)
if teacher_path is not None:
    print(f'Loading saved teacher from {teacher_path}; skipping teacher training.', flush=True)
    saved_teacher = torch.load(teacher_path, map_location='cpu', weights_only=True)
    teacher_cfg = saved_teacher['cfg']
    teacher = WideResNet(teacher_cfg['depth'], teacher_cfg['widen']).to(device)
    # Accept both the original export and the newer checkpoint format.
    state = (saved_teacher['model_state'] if 'model_state' in saved_teacher
             else saved_teacher['state_dict'])
    teacher.load_state_dict(state)
    if 'classes' in saved_teacher:
        assert saved_teacher['classes'] == classes, 'Saved teacher class mapping differs'
    if 'normalization' in saved_teacher:
        assert saved_teacher['normalization'] == dict(mean=MEAN, std=STD), (
            'Saved teacher normalization differs'
        )
    if 'train_indices' in saved_teacher and 'val_indices' in saved_teacher:
        assert saved_teacher['train_indices'] == train_indices.tolist(), 'Teacher training split differs'
        assert saved_teacher['val_indices'] == val_indices.tolist(), 'Teacher validation split differs'
    else:
        print('Legacy teacher has no split metadata; verify it used this same 80/20 split.')
    if 'ids' in saved_teacher:
        assert saved_teacher['ids'] == [row['id'] for row in rows], 'Teacher image IDs differ'
    tb = saved_teacher.get('val_accuracy')
else:
    print('No exported wrn_teacher.pt found; starting teacher training/resume.', flush=True)
    teacher_cfg = dict(TEACH)
    teacher = WideResNet(teacher_cfg['depth'], teacher_cfg['widen']).to(device)
    tb = train(teacher, TEACH['epochs'], TEACH, teacher=None, mixup=False,
               ls=0.1, tag='teacher', resume_checkpoint=TEACHER_RESUME_CHECKPOINT)

# Keep an export in this run's outputs even when loaded from an attached output.
atomic_torch_save(dict(
    model_state=cpu_state(teacher), cfg=teacher_cfg, classes=classes,
    normalization=dict(mean=MEAN, std=STD), val_accuracy=tb,
    # Preserve legacy uncertainty rather than attributing the current split to it.
    **(dict(train_indices=train_indices.tolist(), val_indices=val_indices.tolist(),
            ids=[row['id'] for row in rows]) if teacher_path is None else
       {key: saved_teacher[key] for key in ('train_indices', 'val_indices', 'ids')
        if key in saved_teacher}),
), WORK_DIR / 'wrn_teacher.pt')
teacher.eval()
teacher.requires_grad_(False)
print(f'Frozen teacher ready: {sum(p.numel() for p in teacher.parameters()):,} parameters', flush=True)


Loading saved teacher from /kaggle/input/notebooks/gauravrane2033/wrn-distillation/wrn_teacher.pt; skipping teacher training.
Legacy teacher has no split metadata; verify it used this same 80/20 split.
Frozen teacher ready: 23,354,842 parameters


In [6]:
# Phase 2: distilled student. Its checkpoint must match the frozen teacher.
seed_everything(SEED)
student = SENet(STU['blocks'], STU['channels']).to(device)
ns = cp(student)
print(f'Student parameters: {ns:,}')
assert ns <= 5_000_000, f'Student parameter budget exceeded: {ns:,}'
sb = train(student, STU['epochs'], STU, teacher=teacher, mixup=True,
           ls=0.0, tag='student', resume_checkpoint=STUDENT_RESUME_CHECKPOINT)
# Export the best model with the same metadata used by baseline submission code.
atomic_torch_save(dict(model_state=cpu_state(student), cfg=STU, classes=classes,
                       normalization=dict(mean=MEAN, std=STD), val_accuracy=sb,
                       parameter_count=ns), WORK_DIR / 'wrn_distilled_student.pt')


Student parameters: 4,733,610
[student] Starting a fresh 200-epoch run.
[student] Epoch 001/200: lr=0.010000, train loss=1.4554, val acc=50.86%, best=50.86%, seconds=95.2, phase ETA=315.8 min
[student] Epoch 002/200: lr=0.028000, train loss=1.2805, val acc=60.12%, best=60.12%, seconds=50.1, phase ETA=239.8 min
[student] Epoch 003/200: lr=0.046000, train loss=1.1646, val acc=55.60%, best=60.12%, seconds=50.1, phase ETA=213.9 min
[student] Epoch 004/200: lr=0.064000, train loss=1.0809, val acc=65.23%, best=65.23%, seconds=50.2, phase ETA=200.6 min
[student] Epoch 005/200: lr=0.082000, train loss=1.0172, val acc=66.08%, best=66.08%, seconds=50.2, phase ETA=192.3 min
[student] Epoch 006/200: lr=0.100000, train loss=0.9856, val acc=76.13%, best=76.13%, seconds=50.4, phase ETA=186.6 min
[student] Epoch 007/200: lr=0.099994, train loss=0.9539, val acc=70.27%, best=76.13%, seconds=50.3, phase ETA=182.2 min
[student] Epoch 008/200: lr=0.099974, train loss=0.9089, val acc=71.00%, best=76.13%, se

In [7]:
# Reload the best checkpoint, rather than using the last training epoch.
checkpoint = torch.load(WORK_DIR / 'wrn_distilled_student.pt', map_location=device, weights_only=True)
model = SENet(checkpoint['cfg']['blocks'], checkpoint['cfg']['channels']).to(device)
model.load_state_dict(checkpoint['model_state'])
model.eval()
assert sum(p.numel() for p in model.parameters()) <= 5_000_000

# Convert internal classifier positions back to official CIFAR-10 indices.
CIFAR10_CLASSES = (
    'airplane', 'automobile', 'bird', 'cat', 'deer',
    'dog', 'frog', 'horse', 'ship', 'truck',
)
def official_class_indices(saved_classes):
    names = [str(name) for name in saved_classes]
    if set(names) == {str(i) for i in range(10)}:
        return [int(name) for name in names]
    if set(names) == set(CIFAR10_CLASSES):
        return [CIFAR10_CLASSES.index(name) for name in names]
    raise ValueError(f'Unknown class mapping: {names}; provide the official mapping explicitly.')

class_indices = official_class_indices(checkpoint['classes'])
assert sorted(class_indices) == list(range(10))
normalization = checkpoint['normalization']
test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(normalization['mean'], normalization['std']),
])

SAMPLE_SUBMISSION_CSV = INPUT_ROOT / 'sample_submission.csv'
TEST_IMAGE_DIR = INPUT_ROOT / 'test_images'
assert SAMPLE_SUBMISSION_CSV.is_file(), SAMPLE_SUBMISSION_CSV
assert TEST_IMAGE_DIR.is_dir(), TEST_IMAGE_DIR
with SAMPLE_SUBMISSION_CSV.open(newline='') as f:
    reader = csv.DictReader(f)
    assert reader.fieldnames == ['id', 'label'], 'Expected sample columns: id,label'
    sample_rows = list(reader)
test_ids = [row['id'] for row in sample_rows]
assert test_ids and all(image_id.strip() for image_id in test_ids), 'Empty test IDs'
assert len(set(test_ids)) == len(test_ids), 'Duplicate sample IDs'

def test_image_key(value):
    # Match IDs such as 1, test_00001, or test_00001.png without changing output IDs.
    stem = Path(str(value).strip()).stem
    if stem.startswith('test_'):
        stem = stem[len('test_'):]
    return str(int(stem)) if stem.isdigit() else stem

def ordered_test_paths(image_dir, ids):
    paths = sorted(p for p in image_dir.rglob('*')
                   if p.is_file() and p.suffix.lower() in {'.png', '.jpg', '.jpeg', '.bmp'})
    by_key = {}
    for path in paths:
        key = test_image_key(path.name)
        if key in by_key:
            raise ValueError(f'Ambiguous test image ID: {by_key[key]} and {path}')
        by_key[key] = path
    keys = [test_image_key(image_id) for image_id in ids]
    assert len(set(keys)) == len(keys), 'Sample IDs resolve to duplicate images'
    missing = [image_id for image_id, key in zip(ids, keys) if key not in by_key]
    assert not missing, f'Missing test images: {missing[:5]}'
    assert set(keys) == set(by_key), 'Sample submission does not cover every test image'
    return [by_key[key] for key in keys]

test_paths = ordered_test_paths(TEST_IMAGE_DIR, test_ids)
class TestImages(Dataset):
    def __len__(self):
        return len(test_paths)
    def __getitem__(self, index):
        with Image.open(test_paths[index]) as image:
            image = image.convert('RGB')
            assert image.size == (32, 32), f'Unexpected test image size: {image.size}'
            return test_transform(image)

test_loader = DataLoader(
    TestImages(), batch_size=BATCH_SIZE, shuffle=False, drop_last=False,
    num_workers=NUM_WORKERS, pin_memory=device.type == 'cuda',
)
def views(images):
    padded = F.pad(images, (4, 4, 4, 4), mode='reflect')
    for i, j in [(0, 0), (0, 8), (8, 0), (8, 8), (4, 4)]:
        crop = padded[:, :, i:i+32, j:j+32]
        yield crop
        yield torch.flip(crop, dims=[3])

predictions = []
with torch.inference_mode():
    for images in test_loader:
        images = images.to(device, non_blocking=True)
        logits = torch.zeros((images.size(0), 10), device=device)
        with torch.amp.autocast('cuda'):
            for view in views(images):
                logits += F.softmax(model(view).float(), dim=1)
        assert logits.shape[1] == 10 and torch.isfinite(logits).all(), 'Invalid test logits'
        predictions.extend(class_indices[index] for index in logits.argmax(1).cpu().tolist())
assert len(predictions) == len(test_ids) == len(test_paths)
assert all(type(label) is int and 0 <= label <= 9 for label in predictions)

submission_path = WORK_DIR / 'submission.csv'
with submission_path.open('w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['id', 'label'])
    writer.writerows(zip(test_ids, predictions))

# Reopen the saved file to verify schema, order, coverage, and integer label encoding.
with submission_path.open(newline='') as f:
    reader = csv.DictReader(f)
    assert reader.fieldnames == ['id', 'label']
    submitted_rows = list(reader)
assert [row['id'] for row in submitted_rows] == test_ids
assert [row['label'] for row in submitted_rows] == [str(label) for label in predictions]
print(f'Saved {len(predictions):,} predictions to {submission_path}')
print('Preview:', submitted_rows[:5])
print('Download submission.csv from Kaggle outputs and upload it to the competition page.')


Saved 2,000 predictions to /kaggle/working/submission.csv
Preview: [{'id': '0', 'label': '6'}, {'id': '1', 'label': '3'}, {'id': '2', 'label': '3'}, {'id': '3', 'label': '9'}, {'id': '4', 'label': '3'}]
Download submission.csv from Kaggle outputs and upload it to the competition page.
